# CG-projection enclosure for engineered updated-plate boxes

This notebook is the engineered-box counterpart of `system_updated_plate_solver_v3.ipynb`. It preserves the same 2200-iteration, Gram-corrected CG/projection algorithm and its diagnostics, but is configured for `box_em1-15`, `box_em1-45`, `box_em1-95`, `box_em2-15`, `box_em2-45`, and `box_em2-95`.

Checkpoint histories track the union of the first 100 coordinates and the original notebook's 80 evenly sampled coordinates. A dedicated heatmap reports contraction percentage over iterations for every one of coordinates 1 through 100. The animations remain separate and use exactly the original sparse set of 80 evenly sampled coordinates.

The engineering notebook writes a box only when the requested condition is feasible and fully validated. If any required file is absent, this notebook stops immediately with a clear message instead of substituting an invalid box.


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML, display
from matplotlib.animation import FuncAnimation
from scipy.sparse import diags, load_npz
from scipy.sparse.linalg import cg


data_dir = Path("system_updated_plate")
A = load_npz(data_dir / "A.npz").tocsr()
b = np.loadtxt(data_dir / "b.dat", dtype=float)
x_star = np.loadtxt(data_dir / "soln_x.dat", dtype=float)
engineered_box_names = (
    "box_em1-15", "box_em1-45", "box_em1-95",
    "box_em2-15", "box_em2-45", "box_em2-95",
)
missing_box_paths = [
    data_dir / f"{name}.dat"
    for name in engineered_box_names
    if not (data_dir / f"{name}.dat").is_file()
]
if missing_box_paths:
    missing_names = ", ".join(path.stem for path in missing_box_paths)
    raise FileNotFoundError(
        "Required engineered boxes are unavailable: " + missing_names + ". "
        "Run system_updated_plate_engineer_initial_boxes.ipynb and inspect its "
        "feasibility report; infeasible targets are intentionally not written."
    )
boxes = {
    name: np.loadtxt(data_dir / f"{name}.dat", dtype=float)
    for name in engineered_box_names
}

n = len(b)
if A.shape != (n, n):
    raise ValueError(f"Expected A to have shape {(n, n)}, got {A.shape}.")
if x_star.shape != (n,):
    raise ValueError(f"Expected x_star to have shape {(n,)}, got {x_star.shape}.")
for box_name, box in boxes.items():
    if box.shape != (n, 2):
        raise ValueError(f"{box_name} should have shape {(n, 2)}, got {box.shape}.")
    if np.any(box[:, 0] > box[:, 1]):
        raise ValueError(f"{box_name} has lower bounds above upper bounds.")
    if not np.all((box[:, 0] <= x_star) & (x_star <= box[:, 1])):
        raise ValueError(f"{box_name} does not contain the stored solution.")
    if np.any(np.isclose(np.mean(box, axis=1), x_star, rtol=0.0, atol=1e-15)):
        raise ValueError(f"{box_name} is centered on the solution in a coordinate.")
    if np.ptp(box[:, 1] - box[:, 0]) == 0.0:
        raise ValueError(f"{box_name} does not have coordinate-varying diameters.")

solution_residual = np.linalg.norm(b - A @ x_star)
print(f"system dimension: {n}")
print(f"matrix nonzeros: {A.nnz}")
print(f"stored solution residual: {solution_residual:.6e}")


## Jacobi scaling

With `scale = sqrt(diag(A))`, solve in the coordinates $y=\mathrm{scale}\,x$ and transform the reported enclosure coordinates back to the original variables.

All six engineered files are checked above for valid ordered bounds and containment of the verified solution. No coordinate-wise expansion is applied by this analysis notebook.


In [ ]:
diagonal = A.diagonal()
if np.any(diagonal <= 0):
    raise ValueError("Jacobi scaling requires a positive diagonal.")

scale = np.sqrt(diagonal)
inverse_scale = 1.0 / scale
A_hat = diags(inverse_scale) @ A @ diags(inverse_scale)
b_hat = inverse_scale * b

cg_projection_iterations = 2200
diagnostic_period = 50
projection_qr_tolerance = 1e-10
projection_max_gram_condition = 1e12
projector_row_block_size = 128
max_sparse_coordinates = 80
target_indices = np.arange(min(100, n), dtype=int)

all_coordinates = np.arange(n, dtype=int)
if n <= max_sparse_coordinates:
    sparse_tracked_indices = all_coordinates
else:
    sparse_positions = np.linspace(0, n - 1, max_sparse_coordinates).round().astype(int)
    sparse_tracked_indices = np.unique(sparse_positions)

tracked_indices = np.unique(np.concatenate((target_indices, sparse_tracked_indices)))
target_history_columns = np.searchsorted(tracked_indices, target_indices)
sparse_history_columns = np.searchsorted(tracked_indices, sparse_tracked_indices)
if not np.array_equal(tracked_indices[target_history_columns], target_indices):
    raise RuntimeError("Could not map all target coordinates into tracked histories.")
if not np.array_equal(tracked_indices[sparse_history_columns], sparse_tracked_indices):
    raise RuntimeError("Could not map all sparse coordinates into tracked histories.")

print(f"target coordinates tracked: {len(target_indices)}")
print(f"sparse animation coordinates tracked: {len(sparse_tracked_indices)}")
print(f"unique checkpoint coordinates: {len(tracked_indices)}")
print(f"fixed CG iterations m: {cg_projection_iterations}")
print(f"diagnostic period: {diagnostic_period}")


## Exact fixed-iteration CG-projection enclosure

The implementation below follows the supplied algorithm directly. It records exactly $m$ CG search directions from the initial box center, selects a numerically independent subset, forms $G_m=P_m^T\hat A P_m$, and uses linear solves with $G_m$ in both $\widehat x_m$ and $\Pi_m$.

Column normalization is used only as an algebraically invariant change of basis for stable computation: rescaling selected columns does not change $P_m(P_m^T\hat A P_m)^{-1}P_m^T$. No identity-Gram or exact-conjugacy shortcut is used.

The exact hull radius is $|\Pi_m|d^0$. To avoid storing the dense 21,570-by-21,570 projector, its rows are formed in blocks; every row is still evaluated for the final enclosure. Additional checkpoint calculations evaluate the same formula on the tracked target/sparse row union for diagnostics only.


In [ ]:
from system_updated_plate.cg_projection_enclosure import (
    fixed_iteration_cg_with_directions,
    project_enclosure_rows,
    run_exact_cg_projection_enclosure,
    select_independent_directions,
)


## Run the enclosure

In [ ]:
box_results = {}

for box_name, box in boxes.items():
    ell_hat = scale * box[:, 0]
    u_hat = scale * box[:, 1]

    result = run_exact_cg_projection_enclosure(
        A=A_hat,
        b=b_hat,
        lower=ell_hat,
        upper=u_hat,
        iterations=cg_projection_iterations,
        tracked_indices=tracked_indices,
        diagnostic_period=diagnostic_period,
        qr_tolerance=projection_qr_tolerance,
        max_gram_condition=projection_max_gram_condition,
        row_block_size=projector_row_block_size,
        progress_label=box_name,
    )

    result["box_name"] = box_name
    result["ell_x"] = result["ell"] / scale
    result["u_x"] = result["u"] / scale
    result["x_hat_x"] = result["x_hat"] / scale
    result["radius_x"] = result["radius"] / scale
    result["ell_history_x"] = result["ell_history"] / scale[tracked_indices][None, :]
    result["u_history_x"] = result["u_history"] / scale[tracked_indices][None, :]
    result["projected_center_history_x"] = (
        result["projected_center_history"] / scale[tracked_indices][None, :]
    )
    result["projected_radius_history_x"] = (
        result["projected_radius_history"] / scale[tracked_indices][None, :]
    )
    initial_center = np.mean(box[tracked_indices], axis=1)
    initial_width = box[tracked_indices, 1] - box[tracked_indices, 0]
    initial_radius = 0.5 * initial_width
    width_history = result["u_history_x"] - result["ell_history_x"]
    with np.errstate(divide="ignore", invalid="ignore"):
        result["shrinkage_history"] = np.clip(
            100.0 * (1.0 - width_history / initial_width[None, :]),
            0.0,
            100.0,
        )
        result["projected_radius_ratio_history"] = (
            result["projected_radius_history_x"] / initial_radius[None, :]
        )
        result["center_shift_ratio_history"] = (
            np.abs(result["projected_center_history_x"] - initial_center[None, :])
            / initial_radius[None, :]
        )
        result["coverage_margin_ratio_history"] = (
            result["projected_radius_ratio_history"]
            - result["center_shift_ratio_history"]
        )

    final_ell = result["ell_x"]
    final_u = result["u_x"]
    result["full_contains_solution"] = bool(
        np.all((final_ell <= x_star) & (x_star <= final_u))
    )
    result["tracked_contains_solution"] = bool(
        np.all(
            (result["ell_history_x"][-1] <= x_star[tracked_indices])
            & (x_star[tracked_indices] <= result["u_history_x"][-1])
        )
    )
    result["qr_rank_history"] = np.r_[
        0, [diagnostic["qr_rank"] for diagnostic in result["diagnostics"]]
    ]
    result["retained_rank_history"] = np.r_[
        0, [diagnostic["retained_rank"] for diagnostic in result["diagnostics"]]
    ]
    result["gram_condition_history"] = np.r_[
        1.0, [diagnostic["gram_condition"] for diagnostic in result["diagnostics"]]
    ]
    result["gram_identity_fro_history"] = np.r_[
        0.0, [diagnostic["gram_identity_fro"] for diagnostic in result["diagnostics"]]
    ]
    result["gram_max_offdiag_history"] = np.r_[
        0.0, [diagnostic["gram_max_offdiag"] for diagnostic in result["diagnostics"]]
    ]
    box_results[box_name] = result

    full_initial_width = box[:, 1] - box[:, 0]
    full_final_width = result["u_x"] - result["ell_x"]
    full_shrinkage = np.clip(
        100.0 * (1.0 - full_final_width / full_initial_width), 0.0, 100.0
    )
    print(f"{box_name} fixed CG iterations: {len(result['residual_norms'])}")
    print(f"{box_name} CG residual after m steps: {result['residual_norms'][-1]:.6e}")
    print(f"{box_name} final QR rank: {result['qr_rank']}")
    print(f"{box_name} final retained rank: {result['retained_rank']}")
    print(f"{box_name} final cond(G): {result['gram_condition']:.6e}")
    print(f"{box_name} max projected-center constraint residual: {result['max_constraint_residual']:.6e}")
    print(
        f"{box_name} final mean full-box shrinkage: "
        f"{np.nanmean(full_shrinkage):.6f}%"
    )
    print(
        f"{box_name} full coordinates tightened: "
        f"{np.count_nonzero(full_shrinkage > 1e-10)} / {n}"
    )
    print(
        f"{box_name} final projected-radius ratio (min/median/max): "
        f"{np.min(result['projected_radius_ratio_history'][-1]):.3e} / "
        f"{np.median(result['projected_radius_ratio_history'][-1]):.3e} / "
        f"{np.max(result['projected_radius_ratio_history'][-1]):.3e}"
    )
    print(
        f"{box_name} raw projected intervals containing the original interval: "
        f"{np.count_nonzero(result['coverage_margin_ratio_history'][-1] >= 1.0)} / "
        f"{len(tracked_indices)}"
    )
    print(
        f"{box_name} final full enclosure contains the stored solution: "
        f"{result['full_contains_solution']}"
    )

max_direction_count = max(result["iterations"][-1] for result in box_results.values())
fig, ax = plt.subplots(figsize=(7.5, 5))
for box_name, result in box_results.items():
    ax.plot(
        result["iterations"],
        result["qr_rank_history"],
        marker="o",
        markersize=3,
        label=f"{box_name} QR rank",
    )
    ax.plot(
        result["iterations"],
        result["retained_rank_history"],
        marker="o",
        markersize=3,
        linestyle="--",
        label=f"{box_name} retained rank",
    )
ax.plot(
    [0, max_direction_count],
    [0, max_direction_count],
    color="black",
    linestyle="--",
    linewidth=1,
    label="available directions",
)
ax.set_xlabel("Number of CG directions")
ax.set_ylabel("Numerically selected rank")
ax.set_title("Independent and retained directions")
ax.grid(alpha=0.25)
ax.legend()
plt.show()


## Contraction diagnostics

These plots follow the performance-analysis ideas in the non-simplified `system_303_solver_v3.ipynb`: total width, incremental width reduction, fraction of coordinates tightened, coordinate-wise final bounds, Gram conditioning, and loss of $A$-conjugacy. The final enclosure is computed for every coordinate. Checkpoint histories use the tracked target/sparse row union to avoid repeating the expensive full-projector hull calculation at every diagnostic iteration.

The additional projected-radius diagnostic isolates the main obstruction. For an original interval with center $c_0$ and radius $d_0$, the raw projected interval contains the entire original interval whenever

$$
\frac{d_m^{\mathrm{proj}}-|x_m-c_0|}{d_0} \ge 1.
$$

In that case, intersecting the two intervals cannot contract that coordinate.

In [ ]:
diagnostic_series = {}

for box_name, result in box_results.items():
    box = boxes[box_name][tracked_indices]
    initial_width = box[:, 1] - box[:, 0]
    width_history = result["u_history_x"] - result["ell_history_x"]
    total_width = np.sum(width_history, axis=1)
    total_width_reduction = np.r_[0.0, total_width[:-1] - total_width[1:]]
    width_tolerance = 100.0 * np.finfo(float).eps * max(1.0, float(np.max(initial_width)))
    contracted_fraction = np.mean(
        width_history < initial_width[None, :] - width_tolerance,
        axis=1,
    )
    diagnostic_series[box_name] = {
        "total_width": total_width,
        "total_width_pct": 100.0 * total_width / total_width[0],
        "incremental_reduction_pct": 100.0 * total_width_reduction / total_width[0],
        "contracted_fraction_pct": 100.0 * contracted_fraction,
    }

fig, axes = plt.subplots(2, 2, figsize=(13, 8.5))
for box_name, result in box_results.items():
    iterations = result["iterations"]
    series = diagnostic_series[box_name]
    axes[0, 0].plot(iterations, series["total_width_pct"], marker="o", markersize=3, label=box_name)
    axes[0, 1].semilogy(
        iterations[1:],
        np.maximum(np.abs(series["incremental_reduction_pct"][1:]), np.finfo(float).tiny),
        marker="o",
        markersize=3,
        label=box_name,
    )
    axes[1, 0].plot(iterations, series["contracted_fraction_pct"], marker="o", markersize=3, label=box_name)

    radius_ratio = result["projected_radius_ratio_history"]
    axes[1, 1].semilogy(iterations, np.min(radius_ratio, axis=1), label=f"{box_name} min")
    axes[1, 1].semilogy(iterations, np.median(radius_ratio, axis=1), linestyle="--", label=f"{box_name} median")
    axes[1, 1].semilogy(iterations, np.max(radius_ratio, axis=1), linestyle=":", label=f"{box_name} max")

axes[0, 0].set(title="Total tracked width", ylabel="Percent of initial total width")
axes[0, 1].set(title="Incremental tracked-width reduction", ylabel="Percent of initial total width")
axes[1, 0].set(title="Coordinates materially tightened", ylabel="Tracked coordinates (%)")
axes[1, 1].set(title="Raw projected radius / initial radius", ylabel="Radius ratio (log scale)")
axes[1, 1].axhline(1.0, color="black", linewidth=0.8, linestyle="--", label="ratio = 1")
for ax in axes.flat:
    ax.set_xlabel("CG iteration")
    ax.grid(alpha=0.25, which="both")
    ax.legend(fontsize=8)
fig.suptitle("Updated plate contraction diagnostics on sampled projector rows")
fig.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for box_name, result in box_results.items():
    iterations = result["iterations"]
    axes[0].semilogy(
        iterations,
        result["gram_condition_history"],
        marker="o",
        markersize=3,
        label=box_name,
    )
    axes[1].semilogy(
        iterations,
        np.maximum(result["gram_identity_fro_history"], np.finfo(float).tiny),
        marker="o",
        markersize=3,
        label=f"{box_name} $\\|G-I\\|_F$",
    )
    axes[1].semilogy(
        iterations,
        np.maximum(result["gram_max_offdiag_history"], np.finfo(float).tiny),
        linestyle="--",
        label=f"{box_name} max off-diagonal",
    )
axes[0].set(title="Condition of sampled normalized Gram matrix", ylabel="cond(G)")
axes[1].set(title="Sampled loss of A-conjugacy", ylabel="Error (log scale)")
for ax in axes:
    ax.set_xlabel("CG iteration")
    ax.grid(alpha=0.25, which="both")
    ax.legend(fontsize=8)
fig.suptitle("Gram diagnostics for all selected directions at each checkpoint")
fig.tight_layout()
plt.show()

for box_name, result in box_results.items():
    box = boxes[box_name][tracked_indices]
    initial_lower = box[:, 0]
    initial_width = box[:, 1] - box[:, 0]
    final_lower = result["ell_history_x"][-1]
    final_upper = result["u_history_x"][-1]
    final_shrinkage = result["shrinkage_history"][-1]
    normalized_solution = (x_star[tracked_indices] - initial_lower) / initial_width
    normalized_final_lower = (final_lower - initial_lower) / initial_width
    normalized_final_upper = (final_upper - initial_lower) / initial_width
    final_radius_ratio = result["projected_radius_ratio_history"][-1]
    final_center_shift_ratio = result["center_shift_ratio_history"][-1]
    final_coverage_margin = result["coverage_margin_ratio_history"][-1]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))
    axes[0].plot(tracked_indices, final_shrinkage, marker="o", markersize=3, linewidth=1)
    axes[0].axhline(0, color="black", linewidth=0.8)
    axes[0].set(title="Final coordinate-wise shrinkage", ylabel="Width reduction (%)")

    axes[1].plot(tracked_indices, normalized_final_lower, marker="o", markersize=3, label="final lower")
    axes[1].plot(tracked_indices, normalized_final_upper, marker="o", markersize=3, label="final upper")
    axes[1].scatter(tracked_indices, normalized_solution, s=12, color="black", label="true solution", zorder=3)
    axes[1].axhline(0, color="0.5", linestyle="--", linewidth=1, label="initial lower")
    axes[1].axhline(1, color="0.5", linestyle=":", linewidth=1, label="initial upper")
    axes[1].set(title="Final bounds normalized to initial box", ylabel="Relative position in initial interval")
    axes[1].legend(fontsize=8)

    axes[2].semilogy(tracked_indices, final_radius_ratio, marker="o", markersize=3, label="projected radius / initial radius")
    axes[2].semilogy(tracked_indices, np.maximum(final_center_shift_ratio, np.finfo(float).tiny), marker="o", markersize=3, label="center shift / initial radius")
    axes[2].axhline(1, color="black", linewidth=0.8, linestyle="--")
    axes[2].set(title="Raw projected interval components", ylabel="Ratio (log scale)")
    axes[2].legend(fontsize=8)

    for ax in axes:
        ax.set_xlabel("Original tracked coordinate index")
        ax.grid(alpha=0.25, which="both")
    fig.suptitle(f"{box_name}: coordinate diagnostics")
    fig.tight_layout()
    plt.show()

    materially_contracted = final_shrinkage > 1e-10
    raw_contains_original = final_coverage_margin >= 1.0
    print(f"{box_name} final diagnostic summary")
    print(f"  materially contracted coordinates: {np.count_nonzero(materially_contracted)} / {len(tracked_indices)}")
    print(f"  raw projected interval contains original: {np.count_nonzero(raw_contains_original)} / {len(tracked_indices)}")
    print(f"  coverage-margin ratio min/median/max: {np.min(final_coverage_margin):.3e} / {np.median(final_coverage_margin):.3e} / {np.max(final_coverage_margin):.3e}")
    print(f"  final cond(G): {result['gram_condition_history'][-1]:.3e}")
    print(f"  final ||G-I||_F: {result['gram_identity_fro_history'][-1]:.3e}")
    print(f"  final max off-diagonal: {result['gram_max_offdiag_history'][-1]:.3e}")
    if np.all(raw_contains_original):
        print("  diagnosis: every sampled raw projected interval contains its original interval, so intersection cannot contract it.")
    elif not np.any(materially_contracted):
        print("  diagnosis: no sampled coordinate contracts, although some raw projected intervals do not fully contain the originals; inspect the normalized-bound plot.")
    else:
        print("  diagnosis: contraction occurs on a subset of sampled coordinates; inspect the fraction and coordinate-wise plots.")


## First-100 contraction histories

Each heatmap shows the width-contraction percentage at every checkpoint for every coordinate in $S=\{1,\ldots,100\}$. Thus no coordinate in the requested target set is omitted or sparsely sampled.


In [ ]:
box_count = len(box_results)
column_count = 3
row_count = int(np.ceil(box_count / column_count))
fig, axes = plt.subplots(
    row_count, column_count,
    figsize=(15, 4.2 * row_count),
    sharex=True, sharey=True,
    squeeze=False,
    constrained_layout=True,
)
meshes = []
for ax, (box_name, result) in zip(axes.flat, box_results.items()):
    target_contraction = result["shrinkage_history"][:, target_history_columns].T
    mesh = ax.pcolormesh(
        result["iterations"],
        target_indices + 1,
        target_contraction,
        shading="auto",
        cmap="viridis",
        vmin=0.0,
        vmax=100.0,
    )
    meshes.append(mesh)
    ax.set_title(box_name)
    ax.set_xlabel("CG iteration")
    ax.set_ylabel("Coordinate in S (one-based)")

for ax in axes.flat[box_count:]:
    ax.set_visible(False)
fig.colorbar(
    meshes[0], ax=[ax for ax in axes.flat[:box_count]],
    label="Width contraction (%)", shrink=0.92,
)
fig.suptitle("Contraction histories for every coordinate in S = {1, ..., 100}")
plt.show()


## Residual comparison with standard CG

For each initial box, SciPy's CG and the recorded-direction implementation start from the same scaled box center and are both limited to the algorithm's fixed iteration count $m$. The plotted residual is $\lVert b-Ax_k\rVert_2$.


In [ ]:
standard_cg_residuals = {}

for box_name, box in boxes.items():
    ell_hat = scale * box[:, 0]
    u_hat = scale * box[:, 1]
    x0 = 0.5 * (ell_hat + u_hat)
    residual_history = []

    def record_residual(xk, history=residual_history):
        history.append(float(np.linalg.norm(b_hat - A_hat @ xk)))

    _, standard_cg_info = cg(
        A_hat,
        b_hat,
        x0=x0,
        rtol=0.0,
        atol=0.0,
        maxiter=cg_projection_iterations,
        callback=record_residual,
    )
    if standard_cg_info not in (0, cg_projection_iterations):
        raise RuntimeError(f"Unexpected SciPy CG info for {box_name}: {standard_cg_info}.")
    standard_cg_residuals[box_name] = np.asarray(residual_history)

residual_column_count = 3
residual_row_count = int(np.ceil(len(box_results) / residual_column_count))
fig, axes = plt.subplots(
    residual_row_count, residual_column_count,
    figsize=(15, 4.2 * residual_row_count), sharey=True, squeeze=False,
)
axes = axes.ravel()
for ax, (box_name, result) in zip(axes, box_results.items()):
    standard_residuals = standard_cg_residuals[box_name]
    implementation_residuals = result["residual_norms"]
    ax.semilogy(
        np.arange(1, len(standard_residuals) + 1),
        standard_residuals,
        label="SciPy CG",
    )
    ax.semilogy(
        np.arange(1, len(implementation_residuals) + 1),
        implementation_residuals,
        linestyle="--",
        label="CG-projection implementation",
    )
    ax.set_xlabel("Iteration")
    ax.set_title(box_name)
    ax.grid(alpha=0.25, which="both")
    ax.legend()

for ax in axes[len(box_results):]:
    ax.set_visible(False)
axes[0].set_ylabel(r"Residual norm $\|b-Ax_k\|_2$")
fig.suptitle("Standard CG versus CG-projection residual history")
fig.tight_layout()
plt.show()


## Controllable animation

The retained diagnostic animation shows coordinate-wise box contraction from the exact Gram-corrected formula. Because the plate system is large, the horizontal axis contains the original 80 evenly sampled coordinate indices. The frame slider advances through the diagnostic checkpoints and includes the fixed final iteration $m$.


In [ ]:
def display_coordinate_contraction_animation(results, indices, history_columns, title):
    frame_iterations = np.unique(
        np.concatenate([result["iterations"] for result in results.values()])
    )

    finite_values = np.concatenate([
        result["shrinkage_history"][:, history_columns][np.isfinite(result["shrinkage_history"][:, history_columns])]
        for result in results.values()
    ])
    y_max = max(1.0, float(np.max(finite_values)) * 1.05) if finite_values.size else 1.0

    fig, ax = plt.subplots(figsize=(9, 4.5))
    lines = {}
    for box_name, result in results.items():
        line, = ax.plot(
            indices,
            np.nan_to_num(result["shrinkage_history"][0, history_columns]),
            marker="o",
            markersize=3,
            linewidth=1,
            label=box_name,
        )
        lines[box_name] = line

    label = ax.text(0.02, 0.95, "", transform=ax.transAxes, va="top")
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set(xlim=(indices[0], indices[-1]), ylim=(0, y_max))
    ax.set_xlabel("Original coordinate index (evenly sampled)")
    ax.set_ylabel("Width reduction (% of original width)")
    ax.set_title(title)
    ax.grid(alpha=0.25)
    ax.legend()

    def update(frame):
        direction_count = frame_iterations[frame]
        artists = []
        for box_name, result in results.items():
            history_index = np.searchsorted(
                result["iterations"], direction_count, side="right"
            ) - 1
            lines[box_name].set_ydata(
                np.nan_to_num(result["shrinkage_history"][history_index, history_columns])
            )
            artists.append(lines[box_name])
        label.set_text(f"CG directions {direction_count}")
        return (*artists, label)

    animation = FuncAnimation(
        fig,
        update,
        frames=len(frame_iterations),
        interval=180,
        repeat_delay=1000,
    )
    display(HTML(animation.to_jshtml(fps=5, default_mode="loop")))
    plt.close(fig)
    return animation


### Coordinate contraction

In [ ]:
sparse_coordinate_contraction_animation = display_coordinate_contraction_animation(
    box_results,
    sparse_tracked_indices,
    sparse_history_columns,
    "Engineered updated-plate boxes: sparse coordinate contraction",
)


## Projected bounds before intersection

This animation shows the raw projected lower and upper bounds $\widehat x_m-d_m^{\mathrm{proj}}$ and $\widehat x_m+d_m^{\mathrm{proj}}$ before they are intersected with the initial box. The initial lower and upper bounds are retained as small hollow reference circles. As in the checkpoint diagnostics above, the animation uses the original 80 evenly sampled coordinate indices.


In [ ]:
def display_preintersection_bound_animation(results, initial_boxes, indices, history_columns, title):
    indices = np.asarray(indices, dtype=int)
    if indices.size == 0:
        raise ValueError("indices must not be empty.")

    frame_iterations = np.unique(np.concatenate([
        result["iterations"][1:] for result in results.values()
    ]))
    box_names = list(results)
    column_count = 3
    row_count = int(np.ceil(len(box_names) / column_count))
    fig, axes = plt.subplots(
        row_count, column_count, figsize=(15, 4.5 * row_count),
        sharex=True, squeeze=False, constrained_layout=True,
    )
    axes = axes.ravel()
    fig.suptitle(title)
    iteration_label = axes[0].text(
        0.02, 0.96, "", transform=axes[0].transAxes, va="top"
    )
    artists = {}

    for ax, box_name in zip(axes, box_names):
        result = results[box_name]
        initial_lower = initial_boxes[box_name][indices, 0]
        initial_upper = initial_boxes[box_name][indices, 1]
        raw_lower = (
            result["projected_center_history_x"][:, history_columns]
            - result["projected_radius_history_x"][:, history_columns]
        )
        raw_upper = (
            result["projected_center_history_x"][:, history_columns]
            + result["projected_radius_history_x"][:, history_columns]
        )

        # Hollow circles are the fixed initial-box references.
        ax.scatter(
            indices, initial_lower, s=14, facecolors="none",
            edgecolors="tab:blue", linewidths=0.8, alpha=0.7,
            label="initial lower",
        )
        ax.scatter(
            indices, initial_upper, s=14, facecolors="none",
            edgecolors="tab:orange", linewidths=0.8, alpha=0.7,
            label="initial upper",
        )
        raw_lower_line, = ax.plot(
            indices, raw_lower[1], color="tab:blue", marker="o",
            markersize=3.5, linewidth=0.8, label="raw projected lower",
        )
        raw_upper_line, = ax.plot(
            indices, raw_upper[1], color="tab:orange", marker="o",
            markersize=3.5, linewidth=0.8, label="raw projected upper",
        )

        finite_values = np.concatenate([
            initial_lower[np.isfinite(initial_lower)],
            initial_upper[np.isfinite(initial_upper)],
            raw_lower[1:][np.isfinite(raw_lower[1:])],
            raw_upper[1:][np.isfinite(raw_upper[1:])],
        ])
        if finite_values.size:
            y_min = float(np.min(finite_values))
            y_max = float(np.max(finite_values))
            padding = 0.05 * max(y_max - y_min, abs(y_min), abs(y_max), 1.0)
            ax.set_ylim(y_min - padding, y_max + padding)

        if indices.size == 1:
            ax.set_xlim(indices[0] - 0.5, indices[0] + 0.5)
        else:
            ax.set_xlim(indices[0], indices[-1])
        ax.set_title(box_name)
        ax.set_xlabel("Original coordinate index (evenly sampled)")
        ax.set_ylabel("Bound value")
        ax.grid(alpha=0.25)
        ax.legend(fontsize=8, ncol=2)
        artists[box_name] = {
            "lower_line": raw_lower_line,
            "upper_line": raw_upper_line,
            "raw_lower": raw_lower,
            "raw_upper": raw_upper,
        }

    for ax in axes[len(box_names):]:
        ax.set_visible(False)

    def update(frame):
        direction_count = frame_iterations[frame]
        changed = []
        for box_name, result in results.items():
            history_index = np.searchsorted(
                result["iterations"], direction_count, side="right"
            ) - 1
            box_artists = artists[box_name]
            box_artists["lower_line"].set_ydata(
                box_artists["raw_lower"][history_index]
            )
            box_artists["upper_line"].set_ydata(
                box_artists["raw_upper"][history_index]
            )
            changed.extend([
                box_artists["lower_line"], box_artists["upper_line"]
            ])
        iteration_label.set_text(f"CG directions {int(direction_count)}")
        return (*changed, iteration_label)

    update(0)
    animation = FuncAnimation(
        fig, update, frames=len(frame_iterations), interval=300,
        blit=False, repeat_delay=1000,
    )
    display(HTML(animation.to_jshtml(fps=4, default_mode="loop")))
    plt.close(fig)
    return animation


preintersection_bound_animation = display_preintersection_bound_animation(
    box_results,
    boxes,
    sparse_tracked_indices,
    sparse_history_columns,
    "Engineered updated-plate boxes: sparse raw projected bounds",
)
